# Forecasting Model: 2025-2027 Projections

This notebook builds a driver-based forecasting model with three scenarios (Conservative, Moderate, Optimistic) for 2025-2027.

## Objectives
1. Extract key drivers from FY2024 data
2. Build three scenario models with explicit assumptions
3. Generate 2025-2027 forecasts
4. Perform sensitivity analysis
5. Assess investment attractiveness

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Load cleaned data
df = pd.read_csv('../data/processed/flt_cleaned_data.csv')
df = df.set_index('Fiscal Year')

print("Data loaded successfully")
print("\nHistorical years:", df.index.tolist())

## Key Drivers Extraction

In [ ]:
# Extract FY2024 drivers
fy2024_ttv = df.loc[2024, 'ttv_billion']
fy2024_revenue = df.loc[2024, 'revenue']
fy2024_ebitda = df.loc[2024, 'underlying_ebitda']
fy2024_pat = df.loc[2024, 'profit_loss_after_tax']
fy2024_ocf = df.loc[2024, 'operating_cash_flow']

# Calculate key ratios
revenue_to_ttv_ratio = fy2024_revenue / (fy2024_ttv * 1000000)
ebitda_margin = fy2024_ebitda / fy2024_revenue
pat_to_ebitda_ratio = fy2024_pat / fy2024_ebitda
ocf_to_pat_ratio = fy2024_ocf / fy2024_pat

print("=" * 80)
print("KEY DRIVERS EXTRACTED FROM FY2024")
print("=" * 80)
print(f"TTV: ${fy2024_ttv:.1f}B")
print(f"Revenue: ${fy2024_revenue:,.0f}M")
print(f"Revenue-to-TTV Ratio: {revenue_to_ttv_ratio:.4f}")
print(f"EBITDA Margin: {ebitda_margin:.4f}")
print(f"PAT-to-EBITDA Ratio: {pat_to_ebitda_ratio:.4f}")
print(f"OCF-to-PAT Ratio: {ocf_to_pat_ratio:.4f}")

## Scenario Definitions

In [ ]:
# Define scenarios
scenarios = {
    'Conservative': {
        'ttv_growth': 0.05,
        'margin_improvement': 0.0015,
        'description': 'Slower travel recovery, margin pressure from competition'
    },
    'Moderate': {
        'ttv_growth': 0.0773,
        'margin_improvement': 0.0025,
        'description': 'Continuation of FY24 trends, steady operational improvement'
    },
    'Optimistic': {
        'ttv_growth': 0.10,
        'margin_improvement': 0.0035,
        'description': 'Strong leisure & corporate rebound, successful cost optimization'
    }
}

print("\n" + "=" * 80)
print("SCENARIO DEFINITIONS & ASSUMPTIONS")
print("=" * 80)

for scenario_name, params in scenarios.items():
    print(f"\n{scenario_name}:")
    print(f"  TTV Growth Rate: {params['ttv_growth']*100:.2f}%")
    print(f"  Annual Margin Improvement: {params['margin_improvement']*100:.2f}%")
    print(f"  Rationale: {params['description']}")

## Forecast Generation

In [ ]:
# Build forecast for each scenario
forecast_results = []

for scenario_name, params in scenarios.items():
    current_ttv = fy2024_ttv
    current_margin = ebitda_margin
    
    for year in [2025, 2026, 2027]:
        # TTV forecast
        current_ttv = current_ttv * (1 + params['ttv_growth'])
        
        # Revenue forecast
        current_revenue = current_ttv * 1000000 * revenue_to_ttv_ratio
        
        # EBITDA forecast
        current_margin = current_margin + params['margin_improvement']
        current_ebitda = current_revenue * current_margin
        
        # PAT forecast
        current_pat = current_ebitda * pat_to_ebitda_ratio
        
        # OCF forecast
        current_ocf = current_pat * ocf_to_pat_ratio
        
        # CAPEX (1.5% of revenue)
        current_capex = current_revenue * 0.015
        
        # FCF
        current_fcf = current_ocf - current_capex
        
        forecast_results.append({
            'Year': year,
            'Scenario': scenario_name,
            'TTV (Billion $)': current_ttv,
            'Revenue': current_revenue,
            'Underlying EBITDA': current_ebitda,
            'EBITDA Margin %': current_margin * 100,
            'Profit After Tax (PAT)': current_pat,
            'Net Margin %': (current_pat / current_revenue) * 100,
            'Operating Cash Flow': current_ocf,
            'CAPEX': current_capex,
            'Free Cash Flow': current_fcf
        })

forecast_df = pd.DataFrame(forecast_results)

# Display 2027 forecasts
print("\n" + "=" * 80)
print("2027 FORECAST RESULTS")
print("=" * 80)

for scenario in ['Conservative', 'Moderate', 'Optimistic']:
    data = forecast_df[(forecast_df['Year'] == 2027) & (forecast_df['Scenario'] == scenario)].iloc[0]
    print(f"\n{scenario}:")
    print(f"  Revenue: ${data['Revenue']:,.0f}M")
    print(f"  EBITDA: ${data['Underlying EBITDA']:,.0f}M ({data['EBITDA Margin %']:.1f}%)")
    print(f"  PAT: ${data['Profit After Tax (PAT)']:,.0f}M")
    print(f"  FCF: ${data['Free Cash Flow']:,.0f}M")

## Sensitivity Analysis

In [ ]:
# Sensitivity analysis
print("\n" + "=" * 80)
print("SENSITIVITY ANALYSIS - 2027 IMPACT")
print("=" * 80)

base_pat_2027 = forecast_df[(forecast_df['Year'] == 2027) & (forecast_df['Scenario'] == 'Moderate')]['Profit After Tax (PAT)'].values[0]

# Interest rate sensitivity
interest_impact = 26.0  # $M
print(f"\n1. Interest Rate +1%:")
print(f"   Additional Interest Expense: ${interest_impact:.1f}M")
print(f"   After-tax Impact: ${-interest_impact * 0.75:.1f}M")
print(f"   % Impact on PAT: {(-interest_impact * 0.75 / base_pat_2027) * 100:.1f}%")

# Demand shock
demand_impact = 0.05  # -5%
revenue_impact = forecast_df[(forecast_df['Year'] == 2027) & (forecast_df['Scenario'] == 'Moderate')]['Revenue'].values[0] * demand_impact
ebitda_impact = revenue_impact * 0.22  # 22% EBITDA margin
print(f"\n2. Demand Shock -5%:")
print(f"   Revenue Impact: ${-revenue_impact:,.0f}M")
print(f"   EBITDA Impact: ${-ebitda_impact:,.0f}M")

# Labor cost inflation
labor_cost_impact = 25.4  # $M
print(f"\n3. Labor Cost Inflation +3%:")
print(f"   Additional Labor Costs: ${labor_cost_impact:.1f}M")
print(f"   EBITDA Impact: ${-labor_cost_impact:.1f}M")
print(f"   % Impact on EBITDA: {(-labor_cost_impact / forecast_df[(forecast_df['Year'] == 2027) & (forecast_df['Scenario'] == 'Moderate')]['Underlying EBITDA'].values[0]) * 100:.1f}%")

## Investment Metrics

In [ ]:
# Calculate investment metrics
print("\n" + "=" * 80)
print("2027 INVESTMENT METRICS (BASE CASE)")
print("=" * 80)

base_case_2027 = forecast_df[(forecast_df['Year'] == 2027) & (forecast_df['Scenario'] == 'Moderate')].iloc[0]

# ROIC vs WACC
roic = 0.1798  # 17.98%
wacc = 0.065   # 6.50%
print(f"\nReturn on Invested Capital (ROIC): {roic*100:.2f}%")
print(f"Weighted Average Cost of Capital (WACC): {wacc*100:.2f}%")
print(f"Value Creation Spread: {(roic - wacc)*100:.2f}%")
print(f"Assessment: {'YES - Strong Value Creation' if roic > wacc else 'NO - Value Destruction'}")

# FCF metrics
fcf_yield = (base_case_2027['Free Cash Flow'] / base_case_2027['Revenue']) * 100
print(f"\nFree Cash Flow: ${base_case_2027['Free Cash Flow']:,.0f}M")
print(f"FCF-to-Revenue: {fcf_yield:.2f}%")
print(f"Assessment: {'Strong' if fcf_yield > 10 else 'Moderate'}")

# EPS calculation
shares_outstanding = 200  # Million shares
eps = base_case_2027['Profit After Tax (PAT)'] / shares_outstanding
print(f"\nEstimated EPS (2027): ${eps:.2f}")
print(f"Implied Share Price (15x P/E): ${eps * 15:.2f}")
print(f"Dividend Yield (50% payout): {(eps * 0.5 / (eps * 15)) * 100:.2f}%")

In [ ]:
# Save forecast results
forecast_df.to_csv('../data/processed/2027_forecast_scenarios.csv', index=False)
print("\nForecast results saved to: ../data/processed/2027_forecast_scenarios.csv")